# TraceGen — your retrained checkpoints (Layer 3) vs the paper

`L3_train.sh` trains the six WMM checkpoints of the table from scratch with the released code and data, up to the
step of the paper checkpoint: per-dataset P=128 on Droid (50,000 steps) and on EpicKitchen (30,000), Joint P=128
(46,000), per-dataset P=400 on Droid (80,000) and on EpicKitchen (60,000), and Joint P=400 (68,000). `L3_eval.sh`
evaluates each one on the full test split exactly like Layer 2. This notebook computes the table cells with the
**same metric code as Layer 1** (`tg_l1.py`) and prints, for each cell, the paper value and your retrained value.

Rows whose run has not finished yet are
shown as `pending`. Run `python apps/tracegen/L2_cache.py` once after the evaluations; this notebook then needs a few seconds.

In [ ]:
import numpy as np
import pandas as pd

import tg_l1
import tg_l2

print("L3 outputs:", tg_l2.l3_root())
for (row, env), name in tg_l2.L3_RUNS.items():
    run = tg_l2.finished_run(name, tg_l2.l3_root())
    print(f"{row:22s} {env:5s} {run.name if run else 'pending'}")

In [ ]:
rows = []
for (row, env), name in tg_l2.L3_RUNS.items():
    run = tg_l2.finished_run(name, tg_l2.l3_root())
    l3 = None
    if run is not None:
        l3 = tg_l1.aggregate(tg_l2.run_per_sample(run))
        assert l3["n_valid"] == tg_l1.N_KEPT[env], (row, env, l3["n_valid"])
    for metric in ("MSE", "MAE", "Endp."):
        col = ("Dr " if env == "droid" else "Ep ") + metric
        paper = tg_l1.PAPER[row][tg_l1.COLUMNS.index(col)]
        v = l3[metric] if l3 else np.nan
        rows.append({"row": row, "column": col, "paper": paper, "L3": v, "L3-paper (%)": 100 * (v - paper) / paper})
cmp = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(cmp.to_string(index=False, float_format=lambda v: f"{v:.4f}", na_rep="pending"))

## Table from the retrained checkpoints only (x100)

In [ ]:
wide = cmp.pivot(index="row", columns="column", values="L3").reindex(
    index=list(dict.fromkeys(r for r, _ in tg_l2.L3_RUNS)), columns=tg_l1.COLUMNS)
print(wide.to_string(float_format=lambda v: f"{v:.3f}", na_rep="pending"))

## Does the paper's conclusion hold?

In the paper, per-dataset has lower error than Joint in all six columns, for P=128 and for P=400 (lower is better for
all three metrics). This cell checks whether your retrained checkpoints keep that order.

In [ ]:
for p in ("128", "400"):
    for col in tg_l1.COLUMNS:
        sub = cmp[cmp.column == col].set_index("row")
        a, b = sub.loc[f"per-dataset (P={p})"], sub.loc[f"Joint (P={p})"]
        paper = "per-dataset" if a.paper < b.paper else "Joint"
        l3 = "pending" if np.isnan(a.L3) or np.isnan(b.L3) else ("per-dataset" if a.L3 < b.L3 else "Joint")
        print(f"P={p} {col:9s} lower error  paper: {paper:12s} retrained: {l3}")